##**Project Title**

*Parameter-Efficient Fine-Tuning for Structured Output Generation: Text-to-
SQL with Schema Validity Enforcement*

##**Main Goal**

Can we adapt an LLM efficiently with LoRA, while also improving the reliability of structured output using structure-aware learning and validation?

## 1. Environment Setup

In this section, we install all required dependencies, check GPU availability, mount Google Drive, and prepare project directories.

In [ ]:
!pip install -q -U \
    trl \
    bitsandbytes \
    datasets \
    sqlparse \
    evaluate \
    wandb

!pip uninstall -y trl transformers peft accelerate
!pip install -q "transformers==4.47.1" "peft==0.13.2" "accelerate==1.2.1" sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 40.4 MB/s eta 0:00:00
Found existing installation: trl 1.2.0
Uninstalling trl-1.2.0:
  Successfully uninstalled trl-1.2.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/

In [ ]:
import numpy as np
import pandas as pd
import torch
import transformers
import accelerate
import peft

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)

numpy: 2.0.2
pandas: 2.2.2
torch: 2.10.0+cu128
transformers: 4.47.1
accelerate: 1.2.1
peft: 0.13.2


In [ ]:
import os
import re
import json
import sqlite3
import random
from pathlib import Path

import torch
import pandas as pd
import sqlparse

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Mount Drive and Set Paths

Spider is stored in Google Drive. We mount Drive and copy the dataset locally for faster access during preprocessing and evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#!ls /content/drive/

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/text2sql_project")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

SPIDER_DRIVE_ROOT = Path("/content/drive/MyDrive/CS_512_Project/spider/spider_data")
SPIDER_LOCAL_ROOT = PROJECT_ROOT / "spider_data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if (not SPIDER_LOCAL_ROOT.exists()) or (not (SPIDER_LOCAL_ROOT / "database").exists()):
    if SPIDER_LOCAL_ROOT.exists():
        shutil.rmtree(SPIDER_LOCAL_ROOT)
    !cp -rv "/content/drive/MyDrive/CS_512_Project/spider/spider_data" "/content/text2sql_project/"
else:
    print("Local Spider copy already complete.")

'/content/drive/MyDrive/CS_512_Project/spider/spider_data' -> '/content/text2sql_project/spider_data'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/train_gold.sql' -> '/content/text2sql_project/spider_data/train_gold.sql'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/dev_gold.sql' -> '/content/text2sql_project/spider_data/dev_gold.sql'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/dev.json' -> '/content/text2sql_project/spider_data/dev.json'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/train_others.json' -> '/content/text2sql_project/spider_data/train_others.json'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/train_spider.json' -> '/content/text2sql_project/spider_data/train_spider.json'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/tables.json' -> '/content/text2sql_project/spider_data/tables.json'
'/content/drive/MyDrive/CS_512_Project/spider/spider_data/README.txt' -> '/content/text2sql_project/spider_data/READ

## 3. Load WikiSQL

WikiSQL is used as a quick smoke-test dataset to validate loading, prompt formatting, tokenization, and the basic training pipeline before moving to Spider.

***We initially planned to use WikiSQL for a smoke-test preprocessing stage, but due to current Hugging Face dataset-script compatibility issues, we moved directly to Spider, which is also the primary and more challenging benchmark for the actual project.***

In [ ]:
# !pip install -q "datasets<4.0.0"
# from datasets import load_dataset
# wikisql = load_dataset("Salesforce/wikisql")
# print(wikisql)
# print(wikisql["train"][0])

##4. Load Spider

Load the official Spider train/dev JSON files and schema metadata.

In [ ]:
with open(SPIDER_LOCAL_ROOT / "train_spider.json", "r", encoding="utf-8") as f:
    train_spider = json.load(f)

with open(SPIDER_LOCAL_ROOT / "train_others.json", "r", encoding="utf-8") as f:
    train_others = json.load(f)

with open(SPIDER_LOCAL_ROOT / "dev.json", "r", encoding="utf-8") as f:
    dev_spider = json.load(f)

with open(SPIDER_LOCAL_ROOT / "tables.json", "r", encoding="utf-8") as f:
    tables = json.load(f)

print("train_spider:", len(train_spider))
print("train_others:", len(train_others))
print("dev:", len(dev_spider))
print("tables:", len(tables))
print("\nSample example keys:", train_spider[0].keys())

train_spider: 7000
train_others: 1659
dev: 1034
tables: 166

Sample example keys: dict_keys(['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql'])


## 5. Quick Data Inspection

Inspect a few Spider examples to understand the question, SQL, and database mapping.

In [ ]:
for i in range(3):
    ex = train_spider[i]
    print("=" * 100)
    print("Example:", i)
    print("DB ID:", ex["db_id"])
    print("Question:", ex["question"])
    print("Gold SQL:", ex["query"])

Example: 0
DB ID: department_management
Question: How many heads of the departments are older than 56 ?
Gold SQL: SELECT count(*) FROM head WHERE age  >  56
Example: 1
DB ID: department_management
Question: List the name, born state and age of the heads of departments ordered by age.
Gold SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age
Example: 2
DB ID: department_management
Question: List the creation year, name and budget of each department.
Gold SQL: SELECT creation ,  name ,  budget_in_billions FROM department


## 6. Build Schema Map

Create a lookup from `db_id` to schema metadata so each example can be paired with the correct database structure.

In [ ]:
schema_map = {}

for db in tables:
    schema_map[db["db_id"]] = {
        "db_id": db["db_id"],
        "table_names_original": db["table_names_original"],
        "table_names": db["table_names"],
        "column_names_original": db["column_names_original"],
        "column_names": db["column_names"],
        "column_types": db["column_types"],
        "primary_keys": db["primary_keys"],
        "foreign_keys": db["foreign_keys"],
    }

print("Total schemas:", len(schema_map))
print("Sample db_ids:", list(schema_map.keys())[:5])

Total schemas: 166
Sample db_ids: ['perpetrator', 'college_2', 'flight_company', 'icfp_1', 'body_builder']


## 7. Schema Serialization

Convert each Spider schema into a readable text representation containing tables, columns, primary keys, and foreign keys.

In [ ]:
def serialize_spider_schema(schema):
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]
    primary_keys = schema["primary_keys"]
    foreign_keys = schema["foreign_keys"]

    table_to_cols = {i: [] for i in range(len(table_names))}
    for col_idx, (table_idx, col_name) in enumerate(column_names):
        if table_idx == -1:
            continue
        table_to_cols[table_idx].append((col_idx, col_name))

    lines = ["Tables:"]
    for table_idx, table_name in enumerate(table_names):
        cols = [col_name for _, col_name in table_to_cols[table_idx]]
        lines.append(f"- {table_name}({', '.join(cols)})")

    lines.append("\nPrimary Keys:")
    if primary_keys:
        for pk_idx in primary_keys:
            table_idx, col_name = column_names[pk_idx]
            lines.append(f"- {table_names[table_idx]}.{col_name}")
    else:
        lines.append("- None")

    lines.append("\nForeign Keys:")
    if foreign_keys:
        for src_idx, tgt_idx in foreign_keys:
            src_table_idx, src_col = column_names[src_idx]
            tgt_table_idx, tgt_col = column_names[tgt_idx]
            lines.append(f"- {table_names[src_table_idx]}.{src_col} -> {table_names[tgt_table_idx]}.{tgt_col}")
    else:
        lines.append("- None")

    return "\n".join(lines)

In [ ]:
sample = train_spider[0]
sample_schema = schema_map[sample["db_id"]]
print("DB ID:", sample["db_id"])
print(serialize_spider_schema(sample_schema))

DB ID: department_management
Tables:
- department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
- head(head_ID, name, born_state, age)
- management(department_ID, head_ID, temporary_acting)

Primary Keys:
- department.Department_ID
- head.head_ID
- management.department_ID

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> department.Department_ID


## 8. Locate SQLite Database Files

Each Spider example points to a database through `db_id`. We build the path to the corresponding SQLite file for execution-based validation.

In [ ]:
def get_sqlite_path(spider_root, db_id):
    return spider_root / "database" / db_id / f"{db_id}.sqlite"

sqlite_path = get_sqlite_path(SPIDER_LOCAL_ROOT, sample["db_id"])
print(sqlite_path)
print("Exists:", sqlite_path.exists())

/content/text2sql_project/spider_data/database/department_management/department_management.sqlite
Exists: True


## 9. Prompt Design

Create a consistent instruction format for text-to-SQL generation.

In [ ]:
def format_spider_prompt(example, schema_text):
    return (
        "Generate a valid SQLite SQL query for the given question using only the provided schema.\n"
        "Return only SQL.\n\n"
        f"Schema:\n{schema_text}\n\n"
        f"Question:\n{example['question']}\n\n"
        "SQL:"
    )

prompt = format_spider_prompt(sample, serialize_spider_schema(sample_schema))
print(prompt)
print("\nGold SQL:\n", sample["query"])

Generate a valid SQLite SQL query for the given question using only the provided schema.
Return only SQL.

Schema:
Tables:
- department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
- head(head_ID, name, born_state, age)
- management(department_ID, head_ID, temporary_acting)

Primary Keys:
- department.Department_ID
- head.head_ID
- management.department_ID

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> department.Department_ID

Question:
How many heads of the departments are older than 56 ?

SQL:

Gold SQL:
 SELECT count(*) FROM head WHERE age  >  56


## 10. Preprocessing

Convert raw Spider records into train-ready examples containing prompt text, gold SQL, schema text, and SQLite path.

In [ ]:
def preprocess_spider_example(example, schema_map, spider_root):
    schema_text = serialize_spider_schema(schema_map[example["db_id"]])
    prompt = format_spider_prompt(example, schema_text)
    sqlite_path = get_sqlite_path(spider_root, example["db_id"])

    return {
        "db_id": example["db_id"],
        "question": example["question"],
        "prompt": prompt,
        "target_sql": example["query"].strip(),
        "schema_text": schema_text,
        "sqlite_path": str(sqlite_path),
        "text": prompt + " " + example["query"].strip(),
    }

processed_sample = preprocess_spider_example(sample, schema_map, SPIDER_LOCAL_ROOT)
for k, v in processed_sample.items():
    if isinstance(v, str):
        print(f"\n--- {k} ---\n{v[:800]}")
    else:
        print(k, v)


--- db_id ---
department_management

--- question ---
How many heads of the departments are older than 56 ?

--- prompt ---
Generate a valid SQLite SQL query for the given question using only the provided schema.
Return only SQL.

Schema:
Tables:
- department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
- head(head_ID, name, born_state, age)
- management(department_ID, head_ID, temporary_acting)

Primary Keys:
- department.Department_ID
- head.head_ID
- management.department_ID

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> department.Department_ID

Question:
How many heads of the departments are older than 56 ?

SQL:

--- target_sql ---
SELECT count(*) FROM head WHERE age  >  56

--- schema_text ---
Tables:
- department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
- head(head_ID, name, born_state, age)
- management(department_ID, head_ID, temporary_acting)

Primary Keys:
- department.Depart

In [ ]:
combined_train = train_spider + train_others
processed_train = [preprocess_spider_example(ex, schema_map, SPIDER_LOCAL_ROOT) for ex in combined_train[:2500]]
processed_dev = [preprocess_spider_example(ex, schema_map, SPIDER_LOCAL_ROOT) for ex in dev_spider]

print("Processed train subset:", len(processed_train))
print("Processed dev subset:", len(processed_dev))
print(processed_train[0].keys())

Processed train subset: 2500
Processed dev subset: 1034
dict_keys(['db_id', 'question', 'prompt', 'target_sql', 'schema_text', 'sqlite_path', 'text'])


## 11. Quick Length Analysis

Check approximate prompt and SQL sizes before tokenization.

In [ ]:
prompt_lengths = [len(x["prompt"].split()) for x in processed_train]
sql_lengths = [len(x["target_sql"].split()) for x in processed_train]

print("Prompt word count - min / avg / max:",
      min(prompt_lengths),
      round(sum(prompt_lengths) / len(prompt_lengths), 2),
      max(prompt_lengths))

print("SQL word count - min / avg / max:",
      min(sql_lengths),
      round(sum(sql_lengths) / len(sql_lengths), 2),
      max(sql_lengths))

Prompt word count - min / avg / max: 50 104.91 255
SQL word count - min / avg / max: 4 15.14 61


## 12. Load Base Model

Phi-3.5 Mini Instruct is designed for chat-format prompting. In this section, we load the tokenizer and model with a version-compatible setup and use the tokenizer's chat template for inference.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="eager",
)
model.eval()

print("Model loaded successfully.")
print('Loaded model: ', MODEL_NAME)
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Model loaded successfully.
Loaded model:  microsoft/Phi-3.5-mini-instruct
Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3LongRoPEScaledRotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
   

## 13. Base-Model Inference on Spider

Run the base model on a few Spider examples to establish a qualitative baseline before fine-tuning.

In [ ]:
def build_messages_from_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are a text-to-SQL assistant. Return only valid SQLite SQL."},
        {"role": "user", "content": prompt},
    ]

def generate_sql_chat(prompt, model, tokenizer, max_new_tokens=256):
    messages = build_messages_from_prompt(prompt)

    chat_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    chat_inputs = {k: v.to(model.device) for k, v in chat_inputs.items()}

    if "attention_mask" not in chat_inputs:
        chat_inputs["attention_mask"] = torch.ones_like(chat_inputs["input_ids"])

    prompt_len = chat_inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **chat_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_ids = outputs[0][prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [ ]:
#chat template
test_prompt = processed_train[0]["prompt"]
messages = build_messages_from_prompt(test_prompt)

formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted[:2000])

<|system|>
You are a text-to-SQL assistant. Return only valid SQLite SQL.<|end|>
<|user|>
Generate a valid SQLite SQL query for the given question using only the provided schema.
Return only SQL.

Schema:
Tables:
- department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
- head(head_ID, name, born_state, age)
- management(department_ID, head_ID, temporary_acting)

Primary Keys:
- department.Department_ID
- head.head_ID
- management.department_ID

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> department.Department_ID

Question:
How many heads of the departments are older than 56 ?

SQL:<|end|>
<|assistant|>



In [ ]:
#Checking the output for 1 example

ex = processed_train[0]

print("Question:", ex["question"])
print("Gold SQL:", ex["target_sql"])
print("\nModel output:\n")
print(generate_sql_chat(ex["prompt"], model, tokenizer))

Question: How many heads of the departments are older than 56 ?
Gold SQL: SELECT count(*) FROM head WHERE age  >  56

Model output:



The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


```sql
SELECT COUNT(*) 
FROM management m
JOIN head h ON m.head_ID = h.head_ID
WHERE h.age > 56;
```
This SQL query joins the `management` and `head` tables on the `head_ID` foreign key, and then filters the results to count the number of heads whose age is greater than 56.


In [ ]:
#Checking the output for 3 example


for idx in range(3):
    ex = processed_train[idx]
    print("=" * 100)
    print("Example:", idx)
    print("Question:", ex["question"])
    print("Gold SQL:", ex["target_sql"])
    print("\nModel output:\n")
    print(generate_sql_chat(ex["prompt"], model, tokenizer))

Example: 0
Question: How many heads of the departments are older than 56 ?
Gold SQL: SELECT count(*) FROM head WHERE age  >  56

Model output:

```sql
SELECT COUNT(*) 
FROM management m
JOIN head h ON m.head_ID = h.head_ID
WHERE h.age > 56;
```
This SQL query joins the `management` and `head` tables on the `head_ID` foreign key, and then filters the results to count the number of heads whose age is greater than 56.
Example: 1
Question: List the name, born state and age of the heads of departments ordered by age.
Gold SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age

Model output:

```sql
SELECT head.name, head.born_state, head.age
FROM head
JOIN management ON head.head_ID = management.head_ID
JOIN department ON management.department_ID = department.Department_ID
ORDER BY head.age;
```
This SQL query joins the three tables based on their foreign keys, selects the required columns (name, born state, and age), and orders the result by the age of the heads of departments.
Examp

## 14. Clean Generated SQL

The raw output from a chat model may include extra explanation around the SQL query.  
In this section, we extract only the SQL portion so that later validation and execution checks operate on cleaner model predictions.

In [ ]:
import re

def extract_sql_from_output(model_text):
    """
    Extract the main SQL statement from model output.
    This is useful because chat models often add explanation before or after SQL.
    """
    text = model_text.strip()

    sql_block = re.search(r"```sql\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if sql_block:
        return sql_block.group(1).strip()

    generic_block = re.search(r"```\s*(.*?)```", text, flags=re.DOTALL)
    if generic_block:
        candidate = generic_block.group(1).strip()
        if "select" in candidate.lower():
            return candidate

    select_match = re.search(
        r"(select\b.*?)(?:;|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if select_match:
        return select_match.group(1).strip() + ";"

    return text

## 15. Compare Raw and Cleaned Model Output

Before adding validation, we inspect how much post-processing is needed by comparing raw model responses with cleaned SQL strings.

In [ ]:
for idx in range(3):
    ex = processed_train[idx]
    raw_output = generate_sql_chat(ex["prompt"], model, tokenizer)
    cleaned_output = extract_sql_from_output(raw_output)

    print("=" * 100)
    print("Example:", idx)
    print("Question:", ex["question"])
    print("Gold SQL:", ex["target_sql"])
    print("\nRaw model output:\n", raw_output[:1500])
    print("\nCleaned SQL:\n", cleaned_output)

Example: 0
Question: How many heads of the departments are older than 56 ?
Gold SQL: SELECT count(*) FROM head WHERE age  >  56

Raw model output:
 ```sql
SELECT COUNT(*) 
FROM management m
JOIN head h ON m.head_ID = h.head_ID
WHERE h.age > 56;
```
This SQL query joins the `management` and `head` tables on the `head_ID` foreign key, and then filters the results to count the number of heads whose age is greater than 56.

Cleaned SQL:
 SELECT COUNT(*) 
FROM management m
JOIN head h ON m.head_ID = h.head_ID
WHERE h.age > 56;
Example: 1
Question: List the name, born state and age of the heads of departments ordered by age.
Gold SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age

Raw model output:
 ```sql
SELECT head.name, head.born_state, head.age
FROM head
JOIN management ON head.head_ID = management.head_ID
JOIN department ON management.department_ID = department.Department_ID
ORDER BY head.age;
```
This SQL query joins the three tables based on their foreign keys, selects the 

## 16. SQL Parse Validation

A generated query is only useful if it is structurally valid SQL.  
This section checks whether cleaned model outputs can be parsed successfully.

In [ ]:
import sqlparse

def basic_parse_check(sql_text):
    """
    Return True if sqlparse can parse the SQL into at least one statement.
    """
    try:
        parsed = sqlparse.parse(sql_text)
        return len(parsed) > 0
    except Exception:
        return False

## 17. SQLite Execution Check

Parsing alone is not enough.  
A query may be syntactically valid but still fail against the database schema, so we also test executability on the corresponding SQLite database.

In [ ]:
import sqlite3

def basic_execution_check(sql_text, sqlite_path):
    """
    Try executing SQL against the correct Spider SQLite database.
    Returns a success flag and an optional error message.
    """
    try:
        conn = sqlite3.connect(sqlite_path)
        cur = conn.cursor()
        cur.execute(sql_text)
        _ = cur.fetchall()
        conn.close()
        return True, None
    except Exception as e:
        return False, str(e)

## 18. Small-Scale Base Model Evaluation

We run the base model on a few Spider examples and measure whether the cleaned SQL is parseable and executable.  
This provides a lightweight qualitative baseline before any fine-tuning.

In [ ]:
results = []

for idx in range(5):
    ex = processed_train[idx]
    raw_output = generate_sql_chat(ex["prompt"], model, tokenizer)
    pred_sql = extract_sql_from_output(raw_output)

    parse_ok = basic_parse_check(pred_sql)
    exec_ok, exec_err = basic_execution_check(pred_sql, ex["sqlite_path"])

    results.append({
        "idx": idx,
        "db_id": ex["db_id"],
        "question": ex["question"],
        "gold_sql": ex["target_sql"],
        "pred_sql": pred_sql,
        "parse_ok": parse_ok,
        "exec_ok": exec_ok,
        "exec_err": exec_err,
    })

results_df = pd.DataFrame(results)
results_df

,idx,db_id,question,gold_sql,pred_sql,parse_ok,exec_ok,exec_err
0,0,department_management,How many heads of the departments are older th...,SELECT count(*) FROM head WHERE age > 56,SELECT COUNT(*) \nFROM management m\nJOIN head...,True,True,None
1,1,department_management,"List the name, born state and age of the heads...","SELECT name , born_state , age FROM head ORD...","SELECT head.name, head.born_state, head.age\nF...",True,True,None
2,2,department_management,"List the creation year, name and budget of eac...","SELECT creation , name , budget_in_billions ...","SELECT department.Creation, department.Name, d...",True,True,None
3,3,department_management,What are the maximum and minimum budget of the...,"SELECT max(budget_in_billions) , min(budget_i...",SELECT MAX(Budget_in_Billions) AS Maximum_Budg...,True,True,None
4,4,department_management,What is the average number of employees of the...,SELECT avg(num_employees) FROM department WHER...,SELECT AVG(Num_Employees) AS Average_Employees...,True,True,None


## 19. Failure Analysis

We inspect failed examples to understand whether the errors come from syntax issues, schema mismatches, or extra explanatory text in the model output.

In [ ]:
failed = results_df[(results_df["parse_ok"] == False) | (results_df["exec_ok"] == False)]
failed

,idx,db_id,question,gold_sql,pred_sql,parse_ok,exec_ok,exec_err


## 20. Save Evaluation Results

We save the initial baseline results so they can be reused later for comparison with fine-tuned and reranked models.

In [ ]:
results_path = OUTPUT_DIR / "base_model_eval_sample.csv"
results_df.to_csv(results_path, index=False)

print("Saved to:", results_path)

Saved to: /content/text2sql_project/outputs/base_model_eval_sample.csv


## Save Processed Spider Data to Google Drive

We save the processed train and dev examples so later notebooks can load them directly without repeating preprocessing.

In [ ]:
from google.colab import drive
from pathlib import Path
import json

# drive.mount('/content/drive')

DRIVE_ROOT = Path("/content/drive/MyDrive/CS_512_Project")
PROCESSED_DIR = DRIVE_ROOT / "processed_data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_train_path = PROCESSED_DIR / "processed_train.json"
processed_dev_path = PROCESSED_DIR / "processed_dev.json"

with open(processed_train_path, "w", encoding="utf-8") as f:
    json.dump(processed_train, f, ensure_ascii=False, indent=2)

with open(processed_dev_path, "w", encoding="utf-8") as f:
    json.dump(processed_dev, f, ensure_ascii=False, indent=2)

print("Saved processed_train to:", processed_train_path)
print("Saved processed_dev to:", processed_dev_path)

Saved processed_train to: /content/drive/MyDrive/CS_512_Project/processed_data/processed_train.json
Saved processed_dev to: /content/drive/MyDrive/CS_512_Project/processed_data/processed_dev.json


## 21. Current Progress Summary

At this stage, the project demonstrates:

- successful loading of Spider data and schemas,
- schema serialization into prompt-friendly text,
- prompt construction for text-to-SQL generation,
- base-model inference using Phi-3.5 Mini Instruct,
- post-processing to extract SQL from model outputs,
- initial structural validation via SQL parsing and SQLite execution.
- and saved processed_train and processed_dev for LoRA finetuning and future purposes.

These results establish a working baseline pipeline and prepare the project for LoRA fine-tuning, schema-aware validation, and reranking.